# Baseline CNN, trained on Kaggle

ITRI626 group project. This notebook trains the from-scratch baseline CNN on the shared
70/15/15 split of Stanford Dogs. The logic lives in `Lindani_Model`, not in these cells,
so there is one version of the code rather than one per notebook.

## Before running anything

1. **GPU on.** Right-hand panel, Session options, Accelerator, choose **GPU T4 x2** or
   **GPU P100**. Without it, one epoch takes over an hour instead of a minute or two.
   If the option is greyed out, verify your phone number at kaggle.com/settings.
2. **Add the photographs.** Top right, **Add Input**, search `stanford-dogs-dataset` and
   add *Stanford Dogs Dataset* by Jessica Li.
3. **Add the code.** Add the private dataset holding the contents of `Lindani_Model`
   (see step 2 below). If instead you have working access to GitHub, turn **Internet** on
   in Session options and the notebook will clone the code by itself.

Kaggle allows about 30 GPU hours a week and shuts down idle sessions, so for the long run
use **Save Version, Run All** and let it finish unattended.

## The rule this notebook keeps

The test set is never read during training. It is scored once, afterwards, by
`evaluate.py`. Every reproducibility detail the specification asks for is written to
`run_config.json` automatically rather than reconstructed later.

## Step 1: check the machine

Confirms a GPU is actually attached. If `cuda available` prints `False`, stop and turn the
accelerator on, otherwise everything below runs on the CPU and will be unusably slow.

In [1]:
import torch

print("torch          ", torch.__version__)
print("cuda available ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu            ", torch.cuda.get_device_name(0))
    print("memory gb      ", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

!ls /kaggle/input

torch           2.11.0+cu128
cuda available  True
gpu             Tesla T4
memory gb       14.6
datasets  notebooks


## Step 2: get the code into the session

Two routes, and the cell below picks whichever is available without being told.

**Uploaded code, the route in use.** GitHub is blocked on the development network, so the
code is uploaded to Kaggle as a small private dataset (`train.py`, `model.py`,
`dataset.py`, `split_check.py`, `split_manifest.json` and the leaked photograph list,
about 0.15 MB zipped). Add it to this notebook as an input alongside the dogs dataset.

**Cloning from the repository.** Used automatically if no uploaded code is found. It
takes only the `Lindani_Model` folder through a sparse checkout, rather than the 800 MB
of photographs the repository also holds.

Either way the files are copied into `/kaggle/working/code/Lindani_Model`, because
`/kaggle/input` is **read-only** and the training script writes its results beside the
code.

One caution about the uploaded route: a Kaggle dataset is a snapshot. If any of these
files change on the laptop, the dataset has to be uploaded again, or this notebook will
quietly train yesterday's code.

In [2]:
import shutil
import subprocess
from pathlib import Path

REPO = "https://github.com/JP-NWU-honours-2026/Dog_Classifier.git"
BRANCH = "Lindani-branch"
WORK = "/kaggle/working/code"          # stands in for the repository root
CODE = Path(WORK) / "Lindani_Model"
NEEDED = ["train.py", "model.py", "dataset.py", "split_check.py", "split_manifest.json"]

# Kaggle nests attached inputs differently depending on how they were added, so
# look several folders deep rather than assuming one layout.
DEPTHS = ["*", "*/*", "*/*/*", "*/*/*/*"]


def find_uploaded_code():
    """Looks for the code in any attached dataset, up to four folders deep."""
    root = Path("/kaggle/input")
    for pattern in DEPTHS:
        for folder in sorted(root.glob(pattern)):
            if folder.is_dir() and all((folder / f).is_file() for f in NEEDED):
                return folder
    return None


# Start clean, so a previous half-finished attempt cannot leave stale files behind.
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)

uploaded = find_uploaded_code()

if uploaded:
    print("using uploaded code from", uploaded)
    shutil.copytree(uploaded, CODE, dirs_exist_ok=True)
else:
    print("no uploaded code found, cloning the repository instead")
    subprocess.run(["rm", "-rf", "/kaggle/working/repo"], check=False)
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--depth", "1",
                    "--branch", BRANCH, REPO, "/kaggle/working/repo"], check=True)
    subprocess.run("cd /kaggle/working/repo && git sparse-checkout init --cone "
                   "&& git sparse-checkout set Lindani_Model && git checkout",
                   shell=True, check=True)
    shutil.copytree("/kaggle/working/repo/Lindani_Model", CODE, dirs_exist_ok=True)

print("\ncode now in", CODE)
for path in sorted(CODE.iterdir()):
    print("  ", path.name)

missing = [f for f in NEEDED if not (CODE / f).is_file()]
if missing:
    raise AssertionError(
        f"missing {', '.join(missing)}.\n"
        "If this cell cloned the repository, the branch on GitHub does not have these "
        "files yet, so push them from the laptop and run this cell again. Otherwise add "
        "the code dataset through Add Input."
    )
print("\nAll five required files are present.")

no uploaded code found, cloning the repository instead


Cloning into '/kaggle/working/repo'...


Your branch is up to date with 'origin/Lindani-branch'.

code now in /kaggle/working/code/Lindani_Model
   .gitignore
   DATASET_PROVENANCE.md
   IMPLEMENTATION_GUIDE.md
   MODEL_DESIGN.md
   REPORT_DRAFT.md
   dataset.py
   dataset_figures.py
   duplicate_check.py
   evaluate.py
   kaggle_train.ipynb
   label_audit.py
   model.py
   results
   split_check.py
   split_manifest.json
   train.py

All five required files are present.


## Step 3: find the photographs

`dataset.py` looks for the breed folders in several places and takes the first one it
finds, with the `DOG_IMAGES_DIR` environment variable checked first. The cell below
searches every dataset attached to this notebook for a folder containing at least 100
folders named like `n02085620-Chihuahua`, then sets that variable.

So it does not matter which mirror of Stanford Dogs was added, or how deeply the folders
are nested inside it. It also prints every input attached, which is useful when something
is missing.

If it fails, the message says what to do: add the *Stanford Dogs Dataset* by Jessica Li
through **Add Input**.

In [3]:
import os
from pathlib import Path

MAX_DEPTH = 7


def find_breed_folders():
    """
    Walks the attached inputs looking for the folder that holds the 120 breed
    folders, which are named like n02085620-Chihuahua.

    Kaggle mounts inputs as /kaggle/input/datasets/<owner>/<dataset>/..., so the
    folder sits five levels down rather than one or two. Walking the tree is
    safer than guessing any particular layout, and it stops at the parent of the
    breed folders rather than descending into 20,580 photographs.
    """
    root = Path("/kaggle/input")
    stack = [(root, 0)]
    while stack:
        folder, depth = stack.pop()
        try:
            children = [p for p in folder.iterdir() if p.is_dir()]
        except (PermissionError, OSError):
            continue
        breeds = [p for p in children if p.name.startswith("n0")]
        if len(breeds) >= 100:
            return folder, len(breeds)
        if depth < MAX_DEPTH:
            stack.extend((child, depth + 1) for child in children)
    return None, 0


print("Inputs attached to this notebook:")
for owner in sorted(Path("/kaggle/input").glob("*/*")):
    for dataset in sorted(owner.glob("*")):
        print("  ", dataset.relative_to("/kaggle/input"))

images_dir, breed_count = find_breed_folders()
if not images_dir:
    raise AssertionError(
        "No folder of breed folders found in any attached input. Use Add Input to add "
        "the Stanford Dogs Dataset by Jessica Li."
    )

os.environ["DOG_IMAGES_DIR"] = str(images_dir)
print("\nusing          ", images_dir)
print("breed folders  ", breed_count)

Inputs attached to this notebook:
   datasets/jessicali9530/stanford-dogs-dataset
   notebooks/lindanimdlalose/notebook3be21593ad

using           /kaggle/input/datasets/jessicali9530/stanford-dogs-dataset/images/Images
breed folders   120


## Step 4: check the split and the photographs agree

Runs the self-test in `dataset.py`. It must report 14,355 training, 3,025 validation and
3,200 test photographs, 120 breeds, and pass its label check. If it cannot find the
photographs it will say so plainly.

In [4]:
!cd {WORK} && python Lindani_Model/dataset.py

SELF-TEST

  images_dir                         /kaggle/input/datasets/jessicali9530/stanford-dogs-dataset/images/Images
  split_file                         /kaggle/working/code/Lindani_Model/split_manifest.json
  classes                            120
  train_photographs                  14355
  val_photographs                    3025
  test_photographs                   3200
  leaked_test_photographs_excluded   0
  batch_size                         8
  input_size                         224
  seed                               42

  first three breeds                 ['Chihuahua', 'Japanese Spaniel', 'Maltese Dog']
  last breed                         African Hunting Dog

  one training batch
    images                           (8, 3, 224, 224)
    labels                           (8,), range 57 to 106
    first photograph                 n02099712-Labrador_retriever/n02099712_3980.jpg
    pixel values                     -2.12 to 2.64

  label check                        folder 

## Step 5: smoke test, two epochs over a handful of batches

Proves the whole path works on the GPU before committing an hour to it. Takes a minute or
two.

**Check three things in the output:**

- `Training on cuda` followed by the GPU name, not `cpu`
- `1,631,128 trainable parameters`
- the loss falls between the two epochs

Accuracy here is meaningless. Eight batches teaches the model almost nothing.

In [5]:
!cd {WORK} && python Lindani_Model/train.py --quick --name kaggle_smoke --num-workers 2 --overwrite

Training on cuda (Tesla T4)
1,631,128 trainable parameters, 14,355 training photographs, 120 breeds

  epoch 1/2  learning rate 1.00e-03
    train batch 8/8  loss 4.934  30s
    val batch 8/8  loss 4.716  19s
    train loss 4.934  accuracy 0.78%
    val   loss 4.716  accuracy 0.59%  macro F1 0.0012   best so far

  epoch 2/2  learning rate 5.00e-04
    train batch 8/8  loss 4.879  19s
    val batch 8/8  loss 4.664  2s
    train loss 4.879  accuracy 1.76%
    val   loss 4.664  accuracy 7.03%  macro F1 0.0016   best so far

Best epoch 2, validation macro F1 0.0016, 1.6 minutes total.
Saved to /kaggle/working/code/Lindani_Model/results/runs/kaggle_smoke_quick
Note: the epoch cap was reached rather than early stopping, so the model may still have been improving. Report this, and consider more epochs.


## Step 6: iteration 3, two runs

**Where the experiment stands.** Three runs so far, each judged on validation only:

| Run | Parameters | Best validation macro F1 |
|---|---|---|
| Iteration 1, baseline | 1,631,128 | 0.4969 |
| Iteration 2A, stronger regularisation | 1,631,128 | 0.4944 |
| Iteration 2B, deeper | 4,581,784 | **0.5527** |

Iteration 1's curves showed training loss falling while validation loss flattened, which
suggested overfitting was the binding constraint. **Iteration 2A tested that and found
nothing**: raising dropout to 0.5 and weight decay tenfold changed the result by less than
the run-to-run noise. Iteration 2B tested the opposite idea, that the model lacked the
capacity to make fine distinctions, and gained 5.6 points of macro F1 while overfitting
considerably harder, with a 34 point gap between training and validation accuracy.

So capacity, not restraint, was the limit. Iteration 3 follows that finding in two
directions, one factor each.

**Run A, deeper plus stronger regularisation.** The winning architecture with dropout 0.5
and weight decay 1e-3. Tests whether the deeper model's large overfitting gap can be
converted into accuracy, now that the capacity is there to regularise.

**Run B, deeper with a longer schedule.** The same winning architecture over a 120 epoch
cosine schedule instead of 60. Iteration 2B stopped at epoch 51 under a schedule already
annealing towards zero by epoch 60, so it may have run out of schedule rather than out of
learning. Patience is raised from 5 to 10 for this run only, because a longer schedule
plateaus more often in the middle and patience 5 would end the run prematurely, which
would test nothing. That change is recorded in `run_config.json`.

A longer epoch budget is legitimate here and must be justified in the report: a network
starting from random initialisation needs far longer than one starting from pretrained
weights. Wall clock time is logged per run so the comparison stays visible.

**What is still untouched:** the split, the 224 pixel input, the augmentation and the seed,
all shared with JP and Sulaiman. Changing those would improve the number while destroying
the comparison the project exists to make.

Expect roughly three hours for the two runs together. **The test set is still sealed.**

In [6]:
RUNS = ["iter3_deeper_regularised", "iter3_deeper_long"]

# Run A: the winning deeper architecture, now regularised harder.
!cd {WORK} && python Lindani_Model/train.py --epochs 60 --batch-size 64 --num-workers 2 \
    --name iter3_deeper_regularised --convs-per-block 1,1,1,2,2 \
    --dropout 0.5 --weight-decay 1e-3 --overwrite

# Run B: the same architecture over a 120 epoch cosine schedule, patience raised to 10
# so a mid-schedule plateau does not end the run before the schedule has done its work.
!cd {WORK} && python Lindani_Model/train.py --epochs 120 --batch-size 64 --num-workers 2 \
    --name iter3_deeper_long --convs-per-block 1,1,1,2,2 \
    --patience 10 --overwrite

Training on cuda (Tesla T4)
4,581,784 trainable parameters, 14,355 training photographs, 120 breeds

  epoch 1/60  learning rate 1.00e-03
    train batch 50/225  loss 4.847  107s
    train batch 100/225  loss 4.814  215s
    train batch 150/225  loss 4.789  319s
    train batch 200/225  loss 4.766  425s
    train batch 225/225  loss 4.755  490s
    val batch 48/48  loss 4.720  77s
    train loss 4.755  accuracy 1.72%
    val   loss 4.720  accuracy 2.35%  macro F1 0.0034   best so far

  epoch 2/60  learning rate 9.99e-04
    train batch 50/225  loss 4.622  13s
    train batch 100/225  loss 4.589  26s
    train batch 150/225  loss 4.573  38s
    train batch 200/225  loss 4.547  50s
    train batch 225/225  loss 4.527  56s
    val batch 48/48  loss 4.731  9s
    train loss 4.527  accuracy 3.07%
    val   loss 4.731  accuracy 4.50%  macro F1 0.0166   best so far

  epoch 3/60  learning rate 9.97e-04
    train batch 50/225  loss 4.356  13s
    train batch 100/225  loss 4.322  26s
    train

## Step 7: look at the run before leaving

Prints the last few epochs and the headline figures. Two things to note for the report:
whether **early stopping** triggered or the **epoch cap** was reached, and how far the
training and validation curves diverged, which is how overfitting shows up.

In [7]:
import json

for name in RUNS:
    run = Path(WORK) / "Lindani_Model/results/runs" / name
    if not (run / "run_config.json").is_file():
        print(f"{name}: no results, the run did not finish\n")
        continue

    config = json.loads((run / "run_config.json").read_text())
    print(f"=== {name} ===")
    for key in ["trainable_parameters", "dropout", "weight_decay", "convs_per_block",
                "epochs_planned", "epochs_run", "stopped_early", "epoch_cap_reached",
                "early_stopping", "best_epoch", "best_val_macro_f1", "total_minutes"]:
        print(f"  {key:<22} {config.get(key)}")

    rows = (run / "metrics.csv").read_text().strip().split("\n")
    print("  last three epochs:")
    for line in rows[-3:]:
        print("   ", line)
    print()

print("Earlier runs, for comparison, all on validation:")
print("  iteration 1, baseline              macro F1 0.49688, accuracy 50.51 per cent")
print("  iteration 2A, regularised          macro F1 0.49435")
print("  iteration 2B, deeper               macro F1 0.55270")

=== iter3_deeper_regularised ===
  trainable_parameters   4581784
  dropout                0.5
  weight_decay           0.001
  convs_per_block        [1, 1, 1, 2, 2]
  epochs_planned         60
  epochs_run             57
  stopped_early          True
  epoch_cap_reached      False
  early_stopping         validation macro F1, patience 5
  best_epoch             52
  best_val_macro_f1      0.5602
  total_minutes          70.3
  last three epochs:
    55,2.4471741852423238e-05,0.53826,0.84117,0.83887,1.65112,0.55802,0.55343,65.8,False
    56,1.7037086855465846e-05,0.52804,0.84507,0.84304,1.64705,0.56198,0.559,68.8,False
    57,1.0926199633097158e-05,0.52883,0.84354,0.84123,1.63845,0.5643,0.55932,65.0,False

=== iter3_deeper_long ===
  trainable_parameters   4581784
  dropout                0.3
  weight_decay           0.0001
  convs_per_block        [1, 1, 1, 2, 2]
  epochs_planned         120
  epochs_run             47
  stopped_early          True
  epoch_cap_reached      False
  ea

## Step 8: take the results off Kaggle

Kaggle wipes the session when it ends. Only `/kaggle/working` survives, and only if the
notebook version is saved, so copy the run folder somewhere obvious and download it.

**What to do with each file once it is on your laptop**, in
`Lindani_Model/results/runs/baseline_cnn/`:

| File | What to do |
|---|---|
| `run_config.json` | Commit. This is the reproducibility record |
| `metrics.csv` | Commit. The training and validation curves come from this |
| `val_confusion_best.csv` | Commit. Small, and useful for error analysis |
| `best_model.pth` | **Do not commit.** Weights are ignored by git on purpose. Keep it for `evaluate.py` |

In [8]:
import shutil

destination = Path("/kaggle/working/iteration2_runs")
shutil.rmtree(destination, ignore_errors=True)
destination.mkdir(parents=True)

for name in RUNS:
    run = Path(WORK) / "Lindani_Model/results/runs" / name
    if run.is_dir():
        shutil.copytree(run, destination / name)

for folder in sorted(destination.iterdir()):
    print(folder.name)
    for path in sorted(folder.iterdir()):
        print(f"   {path.name:<26} {path.stat().st_size / 1024:>9.0f} kB")

print("\nDownload the iteration2_runs folder from the Output tab on the right.")
print("Put each run in Lindani_Model/results/runs/<name>/ on the laptop, keeping the")
print("four files only. Commit everything except best_model.pth.")

iter3_deeper_long
   best_model.pth                 17931 kB
   metrics.csv                        4 kB
   run_config.json                    2 kB
   val_confusion_best.csv            28 kB
iter3_deeper_regularised
   best_model.pth                 17931 kB
   metrics.csv                        5 kB
   run_config.json                    2 kB
   val_confusion_best.csv            28 kB

Download the iteration2_runs folder from the Output tab on the right.
Put each run in Lindani_Model/results/runs/<name>/ on the laptop, keeping the
four files only. Commit everything except best_model.pth.
